# 01 — Exploration (Phase 0)

Throwaway notebook: load Kaggle Customer Support on Twitter, profile brands,
pick one, sketch intent clusters. **Ship nothing from here into `src/`.**

Outcome: brand = **AppleSupport** — justification in `DECISION_LOG.md`.

In [ ]:
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw/twcs.csv")
assert RAW.exists(), "Missing data/raw/twcs.csv — download thoughtvector/customer-support-on-twitter via kagglehub"

df = pd.read_csv(
    RAW,
    dtype={
        "tweet_id": str,
        "in_response_to_tweet_id": str,
        "response_tweet_id": str,
    },
)
print(f"rows={len(df):,}")
df.head(2)

## Brand profile

Brand accounts = authors of outbound tweets (`inbound == False`).
Score candidates on volume, English-ness (ASCII ratio), reply substance
(avg length + help-keyword rate), and how many distinct customer messages
they answered.

In [ ]:
brands = df.loc[~df["inbound"], "author_id"].value_counts()
print("Top 15 by outbound volume:")
print(brands.head(15).to_string())

rows = []
for brand in brands.head(15).index:
    out = df[(df["author_id"] == brand) & (~df["inbound"])]
    answered_ids = set(out["in_response_to_tweet_id"].dropna().astype(str))
    sample = out["text"].head(2000)
    ascii_ratio = sample.apply(
        lambda t: sum(c.isascii() for c in str(t)) / max(len(str(t)), 1)
    ).mean()
    help_kw = out["text"].str.contains(
        r"DM|order|refund|tracking|sorry|assist", case=False, na=False
    ).mean()
    rows.append({
        "brand": brand,
        "outbound": len(out),
        "customer_msgs_answered": len(answered_ids),
        "avg_reply_len": round(out["text"].str.len().mean(), 1),
        "ascii_ratio": round(ascii_ratio, 3),
        "help_kw_rate": round(help_kw, 3),
    })

profile = pd.DataFrame(rows)
profile

## Decision snapshot

| Brand | Why considered | Why rejected / kept |
|-------|----------------|---------------------|
| AmazonHelp | Highest volume | Multilingual (JP/EN); harder to defend one taxonomy |
| Uber_Support | Clean English | Many replies are DM/link redirects — weak retrieval targets |
| SpotifyCares | Clean product bugs | Lower volume than Apple |
| **AppleSupport** | ~107k outbound, ~107k answered, ASCII≈1.0, long replies | **Selected** |

In [ ]:
BRAND = "AppleSupport"

out = df[(df["author_id"] == BRAND) & (~df["inbound"])]
answered_ids = set(out["in_response_to_tweet_id"].dropna().astype(str))
cust = df[df["tweet_id"].isin(answered_ids)].copy()
print(f"{BRAND}: outbound={len(out):,} customer_msgs_answered={len(cust):,}")

# Keyword buckets = brainstorm only (formal INTENTS land in config.py after Phase 2)
patterns = {
    "ios_update_bug": r"ios|update|beta",
    "battery_performance": r"battery|drain|charg",
    "connectivity": r"wifi|wi-?fi|bluetooth|signal|cellular|lte",
    "app_crash": r"crash|freeze|frozen|restart|reboot|not working|won.?t open",
    "icloud_account": r"icloud|apple.?id|password|login|sign.?in|2fa|two.?factor",
    "hardware": r"iphone|ipad|macbook|watch|airpods|screen|camera|speaker",
    "purchase_billing": r"bill|refund|purchase|subscription|app.?store|payment",
}
text = cust["text"].fillna("").str.lower()
for label, pat in patterns.items():
    print(f"{label:22s} {int(text.str.contains(pat, regex=True).sum()):,}")

print("\nSample customer messages:")
for t in cust["text"].head(8):
    print(" -", str(t).replace("\n", " ")[:140])

## Next

Phase 1: `tests/test_data_prep.py` → `src/data_prep.py` filtered to `AppleSupport`.